**TÍTULO**

F3N7 — Diagramas de Hovmöller do Pacífico equatorial: SSTA, SSHA e vento

**CONTEXTO**

O diagrama de Hovmöller (longitude × tempo) é a ferramenta clássica para visualizar propagação zonal de sinais equatoriais e distinguir ondas de Kelvin (para leste, rápidas) de Rossby (para oeste, lentas), como no arcabouço de Wheeler & Kiladis (1999).

**DESAFIO**

Hipótese: nos eventos fortes, o aquecimento de superfície em Niño 3.4 é precedido por anomalias de SSH propagando de oeste para leste, com inclinação consistente com a velocidade estimada em F3N6. Objetivos: construir Hovmöllers de SSTA com contornos de SSHA sobrepostos, janelas dos eventos fortes, e anotar a velocidade de fase.

**METODOLOGIA**

SSTA: OISST equatorial (média 2°S–2°N) por longitude, anomalia semanal; domínio local 170°W–30°W (leste da linha de data). SSHA filtrada de F3N6 no domínio 120°E–280°E; sobreposição no trecho comum 170°W–80°W (190°E–280°E). Vento: tau_x_anom (média da caixa) como painel lateral sincronizado — o ERA5 local cobre apenas as caixas do projeto, e isso é declarado no gráfico. Janelas: eventos de classe forte/muito forte do catálogo F3N2 (pico ± 40 semanas).

**RESULTADOS ESPERADOS**

Por evento forte: FigF3N7_hovmoller_<evento> (SSTA sombreada + contornos de SSHA + painel de tau_x, reta da velocidade de fase) e tabelas TabF3N7_hovmoller_ssta_<evento>.csv / TabF3N7_hovmoller_ssha_<evento>.csv.

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. WHEELER, M.; KILADIS, G. N. Convectively Coupled Equatorial Waves: Analysis of Clouds and Temperature in the Wavenumber-Frequency Domain. Journal of the Atmospheric Sciences, v. 56, p. 374-399, 1999. DOI: https://doi.org/10.1175/1520-0469(1999)056<0374:CCEWAO>2.0.CO;2
2. CUI, Y. et al. Oceanic Kelvin waves and their role in ENSO evolution. Journal of Geophysical Research: Oceans, 2025. DOI: https://doi.org/10.1029/2025JC023275
3. HUANG, B. et al. Improvements of the Daily Optimum Interpolation Sea Surface Temperature (DOISST) Version 2.1. Journal of Climate, v. 34, p. 2923-2939, 2021. DOI: https://doi.org/10.1175/JCLI-D-20-0166.1

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
sst_lon = f3.load_oisst_equatorial_weekly()
ssta_lon = f3.lon_anomaly_matrix(sst_lon)
ssta_lon.columns = [float(c) % 360 for c in ssta_lon.columns]  # 190E..330E
ssh = f3.load_ssh_equatorial_weekly()
kelvin = f3.kelvin_bandpass(f3.lon_anomaly_matrix(ssh))
tau = pd.to_numeric(master['tau_x_anom'], errors='coerce')
ssta_nino34 = pd.to_numeric(master['nino34_ssta'], errors='coerce')
catalogo = f3.detect_el_nino_events(ssta_nino34)
fortes = catalogo[catalogo['classe'].isin(['forte', 'muito_forte'])]
if fortes.empty:
    fortes = catalogo.tail(2)
velocidade = f3.fit_phase_speed(f3.phase_speed_from_lags(kelvin, reference_lon=160.0))
print(fortes[['evento', 'classe', 'semana_pico']].to_string(index=False))

 evento      classe semana_pico
EN_1983 muito_forte  1983-01-09
EN_1992       forte  1992-01-26
EN_1997 muito_forte  1997-12-28
EN_2009       forte  2009-12-13
EN_2015 muito_forte  2015-12-20
EN_2023       forte  2023-12-10


In [3]:
for _, ev in fortes.iterrows():
    pico = pd.Timestamp(ev['semana_pico'])
    inicio, fim = pico - pd.Timedelta(weeks=40), pico + pd.Timedelta(weeks=40)
    bloco_ssta = ssta_lon.loc[inicio:fim]
    bloco_ssh = kelvin.loc[inicio:fim]
    bloco_tau = tau.loc[inicio:fim]
    if bloco_ssta.empty:
        print(f"{ev['evento']}: fora do dominio de dados; ignorado")
        continue
    f3.save_table(bloco_ssta, f"TabF3N7_hovmoller_ssta_{ev['evento']}")
    f3.save_table(bloco_ssh, f"TabF3N7_hovmoller_ssha_{ev['evento']}")

    fig, (ax, ax_tau) = plt.subplots(1, 2, figsize=(11, 9), sharey=True,
                                     gridspec_kw={'width_ratios': [4, 1]})
    lons_sst = np.array([float(c) for c in bloco_ssta.columns])
    limite = np.nanmax(np.abs(bloco_ssta.to_numpy()))
    malha = ax.pcolormesh(lons_sst, bloco_ssta.index, bloco_ssta.to_numpy(),
                          cmap='RdBu_r', vmin=-limite, vmax=limite)
    lons_ssh = np.array([float(c) for c in bloco_ssh.columns])
    comum = (lons_ssh >= lons_sst.min()) & (lons_ssh <= lons_sst.max())
    if comum.sum() > 4 and not bloco_ssh.empty:
        ax.contour(lons_ssh[comum], bloco_ssh.index,
                   bloco_ssh.loc[:, bloco_ssh.columns[comum]].to_numpy(),
                   levels=6, colors='k', linewidths=0.5)
    if np.isfinite(velocidade['velocidade_m_s']) and velocidade['velocidade_m_s'] > 0:
        semanas = np.arange(0, 30)
        graus = velocidade['velocidade_m_s'] * semanas * 7 * 86400 / f3.EARTH_DEG_M
        origem = lons_sst.min()
        dentro = origem + graus <= lons_sst.max()
        ax.plot(origem + graus[dentro],
                inicio + pd.to_timedelta(semanas[dentro] * 7, unit='D'),
                color='lime', lw=1.5, label=f"{velocidade['velocidade_m_s']:.1f} m/s (F3N6)")
        ax.legend(loc='upper right', fontsize=7)
    ax.set_xlabel('Longitude (graus E)')
    ax.set_title(f"Hovmoller {ev['evento']} ({ev['classe']}): SSTA sombreada + contornos SSHA")
    fig.colorbar(malha, ax=ax, label='SSTA (C)')
    ax_tau.plot(bloco_tau.to_numpy(), bloco_tau.index, lw=0.8)
    ax_tau.axvline(0, color='k', lw=0.5)
    ax_tau.set_xlabel('tau_x_anom (Pa)\n(media da caixa; ERA5 local)')
    fig.tight_layout()
    f3.save_figure(fig, f"FigF3N7_hovmoller_{ev['evento']}")
    plt.close(fig)

[tabela] TabF3N7_hovmoller_ssta_EN_1983.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_1983.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_1983.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] TabF3N7_hovmoller_ssta_EN_1992.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_1992.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_1992.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] TabF3N7_hovmoller_ssta_EN_1997.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_1997.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_1997.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] TabF3N7_hovmoller_ssta_EN_2009.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_2009.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_2009.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] TabF3N7_hovmoller_ssta_EN_2015.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_2015.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_2015.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] TabF3N7_hovmoller_ssta_EN_2023.csv (81x561) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N7_hovmoller_ssha_EN_2023.csv (81x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N7_hovmoller_EN_2023.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
